In [1]:
"""
Vesuvius competition metric.

Expects standard Kaggle paths and Linux in order to manage dependencies.
"""

import glob
import importlib
import os
import subprocess
import sys
import numpy as np
import pandas as pd
from PIL import Image, ImageSequence


class ParticipantVisibleError(Exception):
    pass


class HostVisibleError(Exception):
    pass


def load_volume(path):
    im = Image.open(path)
    slices = []
    for i, page in enumerate(ImageSequence.Iterator(im)):
        slice_array = np.array(page)
        slices.append(slice_array)
    volume = np.stack(slices, axis=0)
    return volume


def install_dependencies():
    """On Kaggle, the topometrics library must be installed during the run. This function handles the entire process."""
    try:
        import topometrics.leaderboard

        return None
    # The broad exception is necessary as the initial import can fail for multiple reasons.
    except:
        pass

    resources_dir = '/kaggle/input/vesuvius-metric-resources'
    install_dir = '/kaggle/working/topological-metrics-kaggle'

    try:
        subprocess.run(
            f'cd {resources_dir} && uv pip install --no-index --find-links=wheels -r topological-metrics-kaggle/requirements.txt',
            shell=True,
            check=True,
        )
        subprocess.run(f'cd /kaggle/working && cp -r {resources_dir}/topological-metrics-kaggle .', shell=True, check=True)
        subprocess.run(
            f'cd {install_dir} && chmod +x scripts/setup_submodules.sh scripts/build_betti.sh && make build-betti',
            shell=True,
            check=True,
        )
        subprocess.run(
            f'cd {install_dir} && uv pip install -e . --no-deps --no-index --no-build-isolation -v',
            shell=True,
            check=True,
        )
        # Add the new library to Python's path and invalidate caches to ensure it's found.
        sys.path.append('/kaggle/working/topological-metrics-kaggle/src')
        importlib.invalidate_caches()

    except Exception as err:
        raise HostVisibleError(f'Failed to install topometrics library: {err}')


def generate_standard_submission(submission_dir: str) -> None:
    # Dependencies installed here as generate_standard_submission is the first metric function that gets called by the orchestrator.
    submission_tifs = glob.glob(f'{submission_dir}/**/*.tif', recursive=True)
    if len(submission_tifs) == 0:
        submission_tifs = glob.glob('/kaggle/tmp/**/*.tif', recursive=True)
    if len(submission_tifs) == 0:
        raise ParticipantVisibleError('No submission files found')
    df = pd.DataFrame({'tif_paths': submission_tifs})
    df['id'] = df['tif_paths'].apply(lambda x: x.split('/')[-1].split('.')[0])
    os.chdir('/kaggle/working')
    df[['id', 'tif_paths']].to_csv('submission.csv', index=False)


def score_single_tif(
    gt,
    pr,
    surface_tolerance = 2,
    voi_connectivity=26,
    voi_transform='one_over_one_plus',
    voi_alpha=0.3,
    topo_weight=0.3,
    surface_dice_weight=0.35,
    voi_weight=0.35,
):
   

    install_dependencies()
    # The import is here to ensure dependencies are loaded first.
    try:
        # Use a standard import now that the path is reliably set.
        import topometrics.leaderboard
    except Exception as err:
        raise HostVisibleError(f'Failed to import topometrics after installation: {err}')

    score_report = topometrics.leaderboard.compute_leaderboard_score(
        predictions=pr,
        labels=gt,
        dims=(0, 1, 2),
        spacing=(1.0, 1.0, 1.0),  # (z, y, x)
        surface_tolerance=2,  # in spacing units
        voi_connectivity=voi_connectivity,
        voi_transform=voi_transform,
        voi_alpha=voi_alpha,
        combine_weights=(topo_weight, surface_dice_weight, voi_weight),  # (Topo, SurfaceDice, VOI)
        fg_threshold=None,  # None => legacy "!= 0"; else uses "x > threshold"
        ignore_label=2,  # voxels with this GT label are ignored
        ignore_mask=None,  # or pass an explicit boolean mask
    )
    return score_report


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    surface_tolerance: float = 2.0,
    voi_connectivity: int = 26,
    voi_transform: str = 'one_over_one_plus',
    voi_alpha: float = 0.3,
    topo_weight: float = 0.3,
    surface_dice_weight: float = 0.35,
    voi_weight: float = 0.35,
) -> float:
    """Returns the mean per-volume Topological Score, Surface Dice, and VOI Scores."""
    if not solution['tif_paths'].apply(os.path.exists).all():
        raise HostVisibleError('Invalid solution file paths')

    solution['pred_paths'] = submission['tif_paths']
    solution['image_score'] = solution.apply(
        lambda row: score_single_tif(
            row['tif_paths'],
            row['pred_paths'],
            surface_tolerance,
            voi_connectivity=voi_connectivity,
            voi_transform=voi_transform,
            voi_alpha=voi_alpha,
            topo_weight=topo_weight,
            surface_dice_weight=surface_dice_weight,
            voi_weight=voi_weight,
        ),
        axis=1,
    )
    return float(np.mean(solution['image_score']))


In [2]:

# te = score_single_tif(tifffile.imread("/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1004283650.tif"),tifffile.imread("/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1004283650.tif"))

In [3]:
# dir(te)
# print(te.surface_dice)

In [4]:
# import numpy as np
# import tifffile
# import scipy.ndimage as ndi
# from skimage.morphology import remove_small_objects


In [5]:
# import tifffile

# values  = np.load("/kaggle/input/blaaaa/kaggle/working/val_preds.npy")
# path = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1004283650.tif"
# ground_truth = tifffile.imread(path)
# print(values)

In [6]:
# arr = values.squeeze(0)
# print(arr.shape)

In [7]:
# finale = np.load("/kaggle/input/thres0-7/kaggle/working/final.npy")
# print(finale.shape)

In [8]:
# finale_bin = (finale > 0).astype(np.uint8)

In [9]:
# print(f"NEW")
# values, counts = np.unique(finale_bin, return_counts=True)

# for v, c in zip(values, counts):
#     print(f"Value: {v}, Count: {c}")


In [10]:
import tifffile
import cc3d

post = "/kaggle/input/datasets/rajatshedshyal/lazy-runner/kaggle/working/102536988_pred.npy"
pre = "/kaggle/input/nnunet-preds-5/kaggle/working/102536988_pred.npy"
gt = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels/102536988.tif"

post_map = np.load(post)
post_map = (post_map > 0).astype(np.float32)

gt_loaded = tifffile.imread(gt)

pre_map = np.load(pre)
pre_map = (pre_map > 0.3).astype(np.float32)

print(f"pre: {pre_map.shape}, gt: {gt_loaded.shape}, post_map:{post_map.shape}")



print(f"Before POST PROCESSING: ")
values1 = score_single_tif(pr=pre_map, gt=gt_loaded, surface_tolerance=2)
print(f"Score: {values1.score}")
print(f"VOI: {values1.voi}")
print(f"Surface_Dice: {values1.surface_dice}")
print(f"Topo: {values1.topo}") 

print(f"After POST PROCESSING: ")
values = score_single_tif(pr=post_map, gt=gt_loaded, surface_tolerance=2)
print(f"Score: {values.score}")
print(f"VOI: {values.voi}")
print(f"Surface_Dice: {values.surface_dice}")
print(f"Topo: {values.topo}")

pre: (320, 320, 320), gt: (320, 320, 320), post_map:(320, 320, 320)
Before POST PROCESSING: 


Using Python 3.11.13 environment at: /usr
Resolved 34 packages in 52ms
Prepared 5 packages in 534ms
Uninstalled 3 packages in 32ms
Installed 5 packages in 20ms
 - connected-components-3d==3.26.1
 + connected-components-3d==3.26.0
 - imagecodecs==2026.1.14
 + imagecodecs==2025.8.2
 - pybind11==3.0.1
 + pybind11==2.13.6
 + pybind11-global==2.13.6
 + surface-distance==0.1


./scripts/setup_submodules.sh
./scripts/build_betti.sh
Using:
  Python_EXECUTABLE=/usr/local/bin/python
  Python_INCLUDE_DIR=/usr/include/python3.11
  Python_LIBRARY=/usr/lib/x86_64-linux-gnu/libpython3.11.so


CMake Deprecation Warning at CMakeLists.txt:3 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.




-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Python: /usr/local/bin/python (found suitable version "3.11.13", minimum required is "3.7") found components: Interpreter Development.Module Development.Embed
-- Performing Test HAS_FLTO
-- Performing Test HAS_FLTO - Success
-- Found pybind11: /usr/local/include (found version "2.13.6")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Configuring done (2.1s)
-- Generating done (0.0s)
-- Build files have been w

lto-wrapper: warning: using serial compilation of 12 LTRANS jobs


gmake[3]: Leaving directory '/kaggle/working/topological-metrics-kaggle/external/Betti-Matching-3D/build'
[100%] Built target betti_matching
gmake[2]: Leaving directory '/kaggle/working/topological-metrics-kaggle/external/Betti-Matching-3D/build'
gmake[1]: Leaving directory '/kaggle/working/topological-metrics-kaggle/external/Betti-Matching-3D/build'


DEBUG uv 0.6.12
DEBUG Marking explicit source tree for reinstall: `/kaggle/working/topological-metrics-kaggle`
DEBUG Searching for default Python interpreter in search path
DEBUG Found `cpython-3.11.13-linux-x86_64-gnu` at `/usr/local/bin/python` (first executable in the search path)
Using Python 3.11.13 environment at: /usr
DEBUG Acquired lock for `/usr`
DEBUG Using request timeout of 30s
DEBUG Found PEP 621 metadata for /kaggle/working/topological-metrics-kaggle in `pyproject.toml` (topometrics-3d)
DEBUG Solving with installed Python version: 3.11.13
DEBUG Solving with target Python version: >=3.11.13
DEBUG Adding direct dependency: topometrics-3d*
DEBUG Searching for a compatible version of topometrics-3d @ file:///kaggle/working/topological-metrics-kaggle (*)
DEBUG Found static `pyproject.toml` for: topometrics-3d @ file:///kaggle/working/topological-metrics-kaggle
DEBUG No workspace root found, using project root
DEBUG Tried 1 versions: topometrics-3d 1
DEBUG marker environment re

Score: 0.5315707019215712
VOI: VOIReport(voi_total=3.627262322878787, voi_split=1.9647134315222914, voi_merge=1.6625488913564954, voi_score=0.4788862186468818, n_foreground=5540007, connectivity=26)
Surface_Dice: 0.8192329125782766
Topo: TopoReport(toposcore=0.2574300199758858, topoF1_by_dim={0: 0.5054945054945055, 1: 0.0018484288354898336, 2: nan}, counts_by_dim={0: (46, 121, 61), 1: (0, 270, 0), 2: (0, 0, 0)}, dims=[0, 1, 2])
After POST PROCESSING: 
Score: 0.490653918303361
VOI: VOIReport(voi_total=3.9462928728381828, voi_split=1.936290222012923, voi_merge=2.01000265082526, voi_score=0.4578989688381806, n_foreground=4427531, connectivity=26)
Surface_Dice: 0.7664090309141142
Topo: TopoReport(toposcore=0.2071537279668593, topoF1_by_dim={0: 0.4034334763948498, 1: 0.0049261083743842365, 2: nan}, counts_by_dim={0: (47, 172, 61), 1: (0, 101, 0), 2: (0, 0, 0)}, dims=[0, 1, 2])


In [11]:
# import tifffile
# import cc3d

# post = "/kaggle/input/arjun-sir-full-testing/post_102536988_pred.npy"
# pre = post.replace("_repaired.npy", "_pred.npy")
# gt = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels/102536988.tif"

# post_map = np.load(post)
# post_map = (post_map > 0).astype(np.float32)

# gt_loaded = tifffile.imread(gt)

# pre_map = np.load(pre)
# pre_map = (pre_map > 0.3).astype(np.float32)

# print(f"pre: {pre_map.shape}, gt: {gt_loaded.shape}, post_map:{post_map.shape}")



# print(f"Before POST PROCESSING: ")
# values1 = score_single_tif(pr=pre_map, gt=gt_loaded, surface_tolerance=2)
# print(f"Score: {values1.score}")
# print(f"VOI: {values1.voi}")
# print(f"Surface_Dice: {values1.surface_dice}")
# print(f"Topo: {values1.topo}") 

# print(f"After POST PROCESSING: ")
# values = score_single_tif(pr=post_map, gt=gt_loaded, surface_tolerance=2)
# print(f"Score: {values.score}")
# print(f"VOI: {values.voi}")
# print(f"Surface_Dice: {values.surface_dice}")
# print(f"Topo: {values.topo}")

In [12]:
# import tifffile
# import cc3d

# post = "/kaggle/input/arjun-sir-full-testing/post_1029212680_repaired.npy"
# pre = post.replace("_repaired.npy", "_pred.npy")
# gt = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1029212680.tif"

# post_map = np.load(post)
# post_map = (post_map > 0).astype(np.float32)

# gt_loaded = tifffile.imread(gt)

# pre_map = np.load(pre)
# pre_map = (pre_map > 0.3).astype(np.float32)

# print(f"pre: {pre_map.shape}, gt: {gt_loaded.shape}, post_map:{post_map.shape}")



# print(f"Before POST PROCESSING: ")
# values1 = score_single_tif(pr=pre_map, gt=gt_loaded, surface_tolerance=2)
# print(f"Score: {values1.score}")
# print(f"VOI: {values1.voi}")
# print(f"Surface_Dice: {values1.surface_dice}")
# print(f"Topo: {values1.topo}") 

# print(f"After POST PROCESSING: ")
# values = score_single_tif(pr=post_map, gt=gt_loaded, surface_tolerance=2)
# print(f"Score: {values.score}")
# print(f"VOI: {values.voi}")
# print(f"Surface_Dice: {values.surface_dice}")
# print(f"Topo: {values.topo}")

In [13]:
# import tifffile
# import cc3d

# post = "/kaggle/input/arjun-sir-full-testing/post_1033784946_repaired.npy"
# pre = post.replace("_repaired.npy", "_pred.npy")
# gt = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1033784946.tif"

# post_map = np.load(post)
# post_map = (post_map > 0).astype(np.float32)

# gt_loaded = tifffile.imread(gt)

# pre_map = np.load(pre)
# pre_map = (pre_map > 0.3).astype(np.float32)

# print(f"pre: {pre_map.shape}, gt: {gt_loaded.shape}, post_map:{post_map.shape}")



# print(f"Before POST PROCESSING: ")
# values1 = score_single_tif(pr=pre_map, gt=gt_loaded, surface_tolerance=2)
# print(f"Score: {values1.score}")
# print(f"VOI: {values1.voi}")
# print(f"Surface_Dice: {values1.surface_dice}")
# print(f"Topo: {values1.topo}") 

# print(f"After POST PROCESSING: ")
# values = score_single_tif(pr=post_map, gt=gt_loaded, surface_tolerance=2)
# print(f"Score: {values.score}")
# print(f"VOI: {values.voi}")
# print(f"Surface_Dice: {values.surface_dice}")
# print(f"Topo: {values.topo}")

In [14]:
# import tifffile
# import cc3d

# post = "/kaggle/input/arjun-sir-full-testing/post_1059332280_repaired.npy"
# pre = post.replace("_repaired.npy", "_pred.npy")
# gt = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1059332280.tif"

# post_map = np.load(post)
# post_map = (post_map > 0).astype(np.float32)

# gt_loaded = tifffile.imread(gt)

# pre_map = np.load(pre)
# pre_map = (pre_map > 0.3).astype(np.float32)

# print(f"pre: {pre_map.shape}, gt: {gt_loaded.shape}, post_map:{post_map.shape}")



# print(f"Before POST PROCESSING: ")
# values1 = score_single_tif(pr=pre_map, gt=gt_loaded, surface_tolerance=2)
# print(f"Score: {values1.score}")
# print(f"VOI: {values1.voi}")
# print(f"Surface_Dice: {values1.surface_dice}")
# print(f"Topo: {values1.topo}") 

# print(f"After POST PROCESSING: ")
# values = score_single_tif(pr=post_map, gt=gt_loaded, surface_tolerance=2)
# print(f"Score: {values.score}")
# print(f"VOI: {values.voi}")
# print(f"Surface_Dice: {values.surface_dice}")
# print(f"Topo: {values.topo}")

In [15]:
# import tifffile
# import cc3d

# post = "/kaggle/input/arjun-sir-full-testing/post_1061356924_repaired.npy"
# pre = post.replace("_repaired.npy", "_pred.npy")
# gt = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1061356924.tif"

# post_map = np.load(post)
# post_map = (post_map > 0).astype(np.float32)

# gt_loaded = tifffile.imread(gt)

# pre_map = np.load(pre)
# pre_map = (pre_map > 0.3).astype(np.float32)

# print(f"pre: {pre_map.shape}, gt: {gt_loaded.shape}, post_map:{post_map.shape}")



# print(f"Before POST PROCESSING: ")
# values1 = score_single_tif(pr=pre_map, gt=gt_loaded, surface_tolerance=2)
# print(f"Score: {values1.score}")
# print(f"VOI: {values1.voi}")
# print(f"Surface_Dice: {values1.surface_dice}")
# print(f"Topo: {values1.topo}") 

# print(f"After POST PROCESSING: ")
# values = score_single_tif(pr=post_map, gt=gt_loaded, surface_tolerance=2)
# print(f"Score: {values.score}")
# print(f"VOI: {values.voi}")
# print(f"Surface_Dice: {values.surface_dice}")
# print(f"Topo: {values.topo}")

In [16]:
# path = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1004283650.tif" 
# ground_truth = tifffile.imread(path)
# print("New Score is: "+ str(score_single_tif(pr=finale_bin, gt=ground_truth, surface_tolerance=2).score) )


In [17]:

# values  = np.load("/kaggle/input/blaaaa/kaggle/working/val_preds.npy")
# arr = values.squeeze(0)
# print("Without PostProcesss Score is: "+ str(score_single_tif(pr=arr, gt=ground_truth, surface_tolerance=2)) )

In [18]:
# import time
# import tifffile
# import os

# for idx,path in enumerate(["1059332280_pred.npy", "1029212680_pred.npy", "1033784946_pred.npy", "1004283650_pred.npy", "1061356924_pred.npy", "102536988_pred.npy"]):

#     time1 = time.time()

#     g_path = os.path.join("/kaggle/input/thr0-9iter2", "post_"+path)
    
#     post_procssed = np.load(g_path)
#     post_procssed = (post_procssed > 0).astype(np.uint8)

#     ground_dir = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels"
#     pre_post = "/kaggle/input/nnunet-preds-5/kaggle/working"
#     ids = path.replace("_pred.npy", "")

#     print("######################################")
    
#     GT = os.path.join(ground_dir, ids+".tif")
#     print(f"GT: {GT}")
#     ground_truth = tifffile.imread(GT)

#     pre = os.path.join(pre_post, ids+"_pred.npy")
#     print(f"pre_post_path: {pre}")
#     pre_vol = np.load(pre)

#     print(f"Before POST PROCESSING: ")
#     values1 = score_single_tif(pr=pre_vol, gt=ground_truth, surface_tolerance=2)
#     print(f"Score: {values1.score}")
#     print(f"VOI: {values1.voi}")
#     print(f"Surface_Dice: {values1.surface_dice}")
#     print(f"Topo: {values1.topo}")  
    
#     print(f"After POST PROCESSING: ")
#     values = score_single_tif(pr=post_procssed, gt=ground_truth, surface_tolerance=2)
#     print(f"Score: {values.score}")
#     print(f"VOI: {values.voi}")
#     print(f"Surface_Dice: {values.surface_dice}")
#     print(f"Topo: {values.topo}")                 
    
#     time2 = time.time()
    
#     print(f"{idx+1}:TOOK around: {time2-time1} secs")

#     print("######################################")
    
    
# #Apply post process on it okay -> 

In [19]:
# #Visualzie
# %matplotlib inline
# import numpy as np
# import tifffile
# import matplotlib.pyplot as plt
# import ipywidgets as widgets
# from IPython.display import display

# def compare_3d_volume(
#   vol1,
#   vol2,
#   axis=0,
#   cmap="viridis"
# ):
#   # vol1 = tifffile.imread(tiff_path_1)
#   # vol2 = tifffile.imread(tiff_path_2)

#   assert vol1.shape == vol2.shape
#   assert axis in (0, 1, 2)

#   vmin = min(vol1.min(), vol2.min())
#   vmax = max(vol1.max(), vol2.max())

#   def get_slice(vol, idx):
#     if axis == 0:
#       return vol[idx]
#     elif axis == 1:
#       return vol[:, idx, :]
#     else:
#       return vol[:, :, idx]

#   slider = widgets.IntSlider(
#     min=0,
#     max=vol1.shape[axis] - 1,
#     step=1,
#     value=vol1.shape[axis] // 2,
#     description="Slice",
#     continuous_update=False
#   )

#   out = widgets.Output()

#   def update(change=None):
#     idx = slider.value
#     with out:
#       out.clear_output(wait=True)
#       fig, axes = plt.subplots(1, 2, figsize=(12, 6))

#       # REMOVE vmin=vmin, vmax=vmax to let matplotlib auto-scale
#       im1 = axes[0].imshow(get_slice(vol1, idx), cmap=cmap)
#       axes[0].set_title(f"Vol 1 (Auto-scaled) | Slice {idx}")
#       plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
#       axes[0].axis("off")

#       im2 = axes[1].imshow(get_slice(vol2, idx), cmap=cmap)
#       axes[1].set_title(f"Vol 2 (Auto-scaled) | Slice {idx}")
#       plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
#       axes[1].axis("off")

#       plt.show()

#       plt.show()

#   slider.observe(update, names="value")

#   display(widgets.VBox([slider, out]))
#   update()

    

In [20]:
# compare_3d_volume(ground_truth, finale_bin)

In [21]:
# tif1 = tifffile.imread("/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1004283650.tif")
# tif2 = np.load("/kaggle/input/testing-merges-6-vals/post_1004283650_pred.npy")

# # tif1 = (tif1 > 0).astype(np.uint8)
# tif2 = (tif2 > 0).astype(np.uint8)

# compare_3d_volume(tif1, tif2)

In [22]:
# tif1 = np.load("/kaggle/input/nnunet-preds-5/kaggle/working/1004283650_pred.npy")
# tif2 = np.load("/kaggle/input/testing-merges-6-vals/post_1004283650_pred.npy")

# tif1 = (tif1 > 0).astype(np.uint8)
# tif2 = (tif2 > 0).astype(np.uint8)

# compare_3d_volume(tif1, tif2)
